<div align="right"><a href="S08-Notch-Filter.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

Dieses Jupyter Notebook kann durch Klicken auf das Symbol rechts in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) geöffnet werden. Am besten wird es jedoch lokal mit [Jupyter Notebook](https://jupyter.org/) angezeigt (z. B. als Teil der [Anaconda](https://www.anaconda.com/)-Distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/L08-Notch-Filter.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Zurück: [L07 Faltungsmatrix](./L07-Faltungsmatrix.ipynb) | [Aufgabe 08](./A08-Notch-Filter.ipynb) | [Inhalt / Übersicht](./00-Python-Notebooks-Overview-DE.ipynb) | Weiter: [L09 Abtastung und Rekonstruktion](./L09-Abtastung-und-Rekonstruktion.ipynb)

</div>

---


## Übung: Notch-Filter (2.X)

Diese Übung soll ein Beispiel für das Design eines rekursiven Notch-Filters darstellen.  
Seine Wirkung wird anhand einiger Beispiele veranschaulicht.

Ein Notch-Filter 2. Ordnung wird im Z-Bereich entworfen.  
Er besitzt konjugiert komplexe Nullstellen auf dem Einheitskreis — entsprechend der Frequenz $\Omega_0$, die unterdrückt werden soll —  
und Pole mit denselben Phasenwinkeln, die geringfügig innerhalb des Einheitskreises liegen.

**Nullstellen:** $e^{\pm j \Omega_0}$  
**Pole:** $\beta e^{\pm j \Omega_0}$

### a)
Schreiben Sie eine Python-Funktion `notchdesign()`, um die Filterkoeffizienten  
$\mathbf{a} = \{a_0, a_1, a_2\}$ und $\mathbf{b} = \{b_0, b_1, b_2\}$ zu berechnen.  
Diese Funktion soll die normierte Frequenz $\Omega_0$ und $\beta$ als Parameter verwenden.  
Berechnen Sie die Koeffizienten entweder analytisch oder mithilfe von `np.poly()`.

### b)
Stellen Sie den Frequenzgang (in dB) eines Notch-Filters dar  
($\Omega_N = \pi/4$, $\beta = 0.9$) mit der Funktion `scipy.signal.freqz()`.

Wie verändert sich der Frequenzgang, wenn der Faktor $\beta$ auf $0.9999$ erhöht wird?  
Überprüfen Sie Ihre Vermutung mit Python.

### c)
Diese Aufgabe demonstriert die Wirkung eines Notch-Filters anhand eines Audio-Beispiels.

Erzeugen Sie ein Testsignal $x(k) = x_1(k) + x_2(k)$, das aus folgenden Komponenten besteht:

| Komponente | Beschreibung |
|-------------|--------------|
| **1. Sinuskomponente** $x_1(k)$ | |
| Frequenz | $f_{sin} = 1000 \text{ Hz}$ |
| Amplitude | $-0.5 \dots 0.5$ |
| **2. Rauschkomponente** $x_2(k)$ | |
| Verteilung | gleichverteilt zwischen $-0.25$ und $0.25$ |

Verwenden Sie eine Abtastfrequenz von $f_a = 8000 \text{ Hz}$.  
Das Testsignal soll eine Dauer von $5\,\mathrm{s}$ haben.

Zeigen Sie das Testsignal $x(k)$ in Python mit `IPython.display.Audio` an und spielen Sie es mit dem Audioplayer des Notebooks ab.  
Verwenden Sie einen Notch-Filter ($\beta = 0.9$), um das sinusförmige Signal $x_1(k)$ in $x(k)$ zu unterdrücken.  
Es sollte nur das Rauschsignal übrig bleiben.

Zum Filtern können Sie die Funktion `scipy.signal.lfilter()` verwenden.  
Zeigen Sie das gefilterte Signal mit `IPython.display.Audio` an und spielen Sie es ab, um den Effekt des Notch-Filters anzuhören.

### d)
Wiederholen Sie das Audio-Beispiel aus Aufgabe (c).  
Verwenden Sie nun einen Notch-Filter mit $\beta = 0.9999$.  
Was hat sich im Vergleich zum Fall $\beta = 0.9$ verändert?  
Was ist die Ursache für diese Veränderung?

Überprüfen Sie das beobachtete Verhalten, indem Sie die Impulsantworten der Notch-Filter mit $\beta = 0.9$ und $\beta = 0.9999$ darstellen.  
Sie können dazu `scipy.signal.lfilter()` und `np.arange()` verwenden, um die Impulsantworten zu erzeugen und zu plotten.

### e) Zusatzaufgabe — praktisches Beispiel
Beim Empfang von Kurzwellenprogrammen können Verzerrungen durch sinusförmige Störsignale auftreten.  
Dies können Träger anderer Kurzwellenstationen sein.

Zur Demonstration dieses Verhaltens gibt es die Datei  
`l936.28_kHz_25.04.2001_23_17.wav`.  
Das Störsignal wurde bei etwa $f = 345 \text{ Hz}$ gemessen.

Ihre Aufgabe besteht nun darin, die Störung mit einigen Zeilen Python-Code zu eliminieren  
und das bereinigte Signal über einen Lautsprecher auszugeben.

**Hinweise:**
- Verwenden Sie die Python-Funktionen `scipy.io.wavfile.read()` und `scipy.io.wavfile.write()` zum Einlesen und Speichern der WAV-Datei.  
- Wählen Sie $\beta$ gemäß Ihrer Erfahrungen aus den Aufgaben (b) und (d).  
- Skalieren Sie das gefilterte Signal so, dass es eine maximale Amplitude von 1 hat.  
- Abhängig von Ihrem System müssen Sie eventuell ein separates Programm verwenden, um die Ausgabedatei abzuspielen,  
  wenn die Abtastrate (z. B. $44{,}1 \text{ kHz}$) nicht unterstützt wird.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import lnotchdesign
from IPython.display import Audio, display
# BEGIN SOLUTION
# ----- Part b: Dimensioning and Frequency Response ----------------------------
Omega0 = np.pi / 4

for beta in [0.9, 0.9999]:
    a, b = lnotchdesign(Omega0, beta)
    w, H = signal.freqz(b, a)

    plt.figure()
    plt.plot(w / np.pi, 20 * np.log10(np.maximum(np.abs(H), 1e-12)))
    plt.title(rf'Frequency Response $\beta$={beta}')
    plt.xlabel(r'$\Omega / \pi$')
    plt.ylabel(r'$|H(e^{j\Omega})|$ in dB')
    plt.ylim([-60, 10])
    plt.grid(True)

input("Continuing with Part c) — press Enter...")

# ----- Part c: Audio Demonstration --------------------------------------------
fa = 8000      # Sampling frequency
fsinus = 1000  # Sinusoid frequency
duration = 5   # seconds
N = int(fa * duration)

# Generate noisy sinusoidal signal
x1 = 0.5 * np.sin(2 * np.pi * fsinus / fa * np.arange(1, N + 1))
x2 = (np.random.rand(N) - 0.5) / 2.0
x = x1 + x2

print("Audio: x(k) unfiltered")
display(Audio(x, rate=fa, normalize=False))

# Filter with beta = 0.9
beta = 0.9
Omega0 = 2 * np.pi * fsinus / fa
a, b = lnotchdesign(Omega0, beta)
y = signal.lfilter(b, a, x)

print(f"Audio: y(k) filtered beta={beta}")
display(Audio(np.clip(y, -1, 1), rate=fa, normalize=False))

input("Continuing with Part d) — press Enter...")

# ----- Part d: Audio Demonstration, Impulse Response Length -------------------
print("Audio: x(k) unfiltered")
display(Audio(x, rate=fa, normalize=False))

beta = 0.9999
a, b = lnotchdesign(Omega0, beta)
y = signal.lfilter(b, a, x)

print(f"Audio: x(k) filtered beta={beta}")
display(Audio(np.clip(y, -1, 1), rate=fa, normalize=False))

for beta in [0.9, 0.9999]:
    a, b = lnotchdesign(Omega0, beta)

    # Impulse response (N samples, sampling rate fa)
    imp = np.zeros(N, dtype=float)
    imp[0] = 1.0
    h = signal.lfilter(b, a, imp)
    t = np.arange(N) / fa

    # Downsample for plotting (every 10th sample)
    t_plot = t[::10]
    h_plot = h[::10]

    plt.figure()
    plt.stem(t_plot, h_plot)
    plt.axis([0, duration, -1e-4, 1e-4])
    plt.title(rf'Impulse Response $\beta$={beta}')
    plt.xlabel(r'$t = k\,T_A$ in s')
    plt.ylabel('h(k)')

plt.show()
# END SOLUTION

## Lösung: Notch-Filter (2.X)

### a)
Nullstellen: $e^{\pm j \Omega_N}$  
Pole: $\beta e^{\pm j \Omega_N}$

Die Übertragungsfunktion $H(z)$ – und damit die Filterkoeffizienten – ergeben sich durch Multiplikation:

$$
\begin{array}{rcl}
H(z) & = & \frac{\left( z -  e^{+ j \Omega_N} \right) \left( z -  e^{- j \Omega_N} \right)}
{ \left( z - \beta e^{+ j \Omega_N} \right) \left( z - \beta e^{- j \Omega_N} \right)} \\
     & = &\frac{z^2 - z \left( e^{+ j \Omega_N} +  e^{- j \Omega_N} \right) + 1}
{z^2 - z \beta \left( e^{+ j \Omega_N} +  e^{- j \Omega_N} \right) + \beta^2} \\
     & = &\frac{z^2 - 2 z \cos\left( \Omega_N \right) + 1}
{z^2 - 2 \beta z \cos\left( \Omega_N \right) + \beta^2} \\
     & = &\frac{1 - 2 z^{-1} \cos\left( \Omega_N \right) + z^{-2}}
{1 - 2 \beta z^{-1} \cos\left( \Omega_N \right) + \beta^2 z^{-2}}
\end{array}
$$

Mit der üblichen Definition der Koeffizientenpaare $\mathbf{b}$ und $\mathbf{a}$ gilt:

$$
H(z) = \frac{\sum\limits_{\mu=0}^{2} b_\mu z^{-\mu}}{1 + \sum\limits_{\nu=1}^{2} a_\nu z^{-\nu}}
$$

Damit ergeben sich die Werte:

$$
\mathbf{b} = \{1, -2\cos(\Omega_N), 1\}
$$

$$
\mathbf{a} = \{1, -2\beta\cos(\Omega_N), \beta^2\}
$$

### b)
Es zeigt sich deutlich, dass die Kerbe im Frequenzbereich schmaler wird,  
wenn die Pole (über den Faktor $\beta$) näher am Einheitskreis liegen.

<img src="./pngs/ldsv2a19_b1.png" width="250">
<img src="./pngs/ldsv2a19_b2.png" width="250">

### c)
...Anhören und genießen...

### d)
Man hört, dass der Notch-Filter mit $\beta = 0.9999$ sehr langsam zu wirken beginnt.  
Dies liegt an der extrem langen Einschwingzeit des Filters.

Dieses Einschwingverhalten zeigt sich auch in der Impulsantwort.

<img src="./pngs/ldsv2a19_d1.png" width="250">
<img src="./pngs/ldsv2a19_d2.png" width="250">

**Erklärung:**  
Da das Signal im Frequenzbereich sehr schmalbandig ist, muss es im Zeitbereich entsprechend länger sein.  
(Dies entspricht dem Unschärfeprinzip der Signalverarbeitung.)

### e)
Mit passend gewählten Koeffizienten kann die Störung nahezu vollständig beseitigt werden.